In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import random
import time

random.seed(42)

In [2]:
def gauss(A, b, precision):
  A = A.astype(precision)
  b = b.astype(precision)
  n = A.shape[0]
  for i in range(n):
    
    _max_i = np.argmax(np.abs(A[i:,i])) + i
    A[[i, _max_i], :] = A[[_max_i, i], :]
    b[i], b[_max_i] = b[_max_i], b[i]
    
    b[i] /= A[i,i]
    A[i,:] /= A[i,i]
    
    for j in range(n):
      if(j != i):
        b[j] -= b[i] * A[j,i]
        A[j,:] -= A[i,:] * A[j,i]
        
  return b

In [3]:
def thomas(A, b, precision):
  A = A.astype(precision)
  b = b.astype(precision)
  n = A.shape[0]
  
  b[0] /= A[0,0]
  A[0,:] /= A[0,0]
  for i in range(1,n):
    b[i] -= b[i-1] * A[i,i-1]
    A[i,:] -= A[i-1,:] * A[i,i-1]
    b[i] /= A[i,i]
    A[i,:] /= A[i,i]
    
  for i in range(n-2,-1,-1):
    b[i] -= b[i+1] * A[i,i+1]
    
  return b

Benchmark metod

In [4]:
def maximum_error(A,B):
  n = len(A)
  return max([np.abs(A[i] - B[i]) for i in range(n)])

In [5]:
def x_vector(n, precision):
  random.seed(42)
  X = np.array([1 if random.randint(1,2) == 1 else -1 for _ in range(n)], dtype=precision)
  return X

def task_1_matrix(n, X, precision):
  A = np.array([[1/(i+j+2-1) if i != 0 else 1 for j in range(n)] for i in range(n)], dtype=precision)
  b = A @ X
  return A, b 

def task_2_matrix(n, X, precision):
  A = np.array([[(2*(i+1))/(j+1) if j >= i else (2*(j+1))/(i+1) for j in range(n)] for i in range(n)], dtype=precision)
  b = A @ X
  return A, b 
  
k, m = 4, 3

def element_tridiagonal(i, j, m, k):
    i1 = i + 1
    j1 = j + 1
    if j1 == i1:
        return -m * i1 - k
    elif j1 == i1 + 1:
        return float(i1)
    elif j1 == i1 - 1 and i1 > 1:
        return m / i1
    else:
        return 0.0

def task_3_matrix(n, X, precision):
    A = np.array([[element_tridiagonal(i, j, m, k) for j in range(n)] for i in range(n)],dtype=precision)
    b = A @ X
    return A, b

Wizualizacja

Kod do obslugi wizualizacji zadań wygenerowano za pomocą llm

Zadanie 1

In [6]:
FIRST_N = list(range(2, 21))
SECOND_N = list(range(2, 200))
PRECISIONS = [np.float32, np.float64]

os.makedirs('data', exist_ok=True)
os.makedirs('plots', exist_ok=True)
pd.set_option('display.float_format', '{:.5f}'.format)


def benchmark(matrix_func, n_range, precisions, method, task_name):
    for precision in precisions:
        precision_name = np.dtype(precision).name
        rows = []

        for n in n_range:
            X = x_vector(n, precision)
            A, b = matrix_func(n, X, precision)

            t0 = time.perf_counter()
            X_approx = method(A.copy(), b.copy(), precision)
            solve_time = time.perf_counter() - t0

            error = maximum_error(X, X_approx)
            cond = np.linalg.cond(A, p=1)
            rows.append({'n': n, 'error': error, 'condition_number': cond, 'time': solve_time})

            print(f"Task: {task_name}, Precision: {precision_name}, n: {n}, "
                  f"Error: {error:.6e}, Cond: {cond:.6e}, Time: {solve_time:.6f}s")

        path = f"data/{task_name}_{precision_name}.csv"
        pd.DataFrame(rows).to_csv(path, index=False, float_format='{:.4e}'.format)
        print(f"Results saved to {path}")


def make_plot(x32, y32, x64, y64, ylabel, title, path, semilogy=False):
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_fn = ax.semilogy if semilogy else ax.plot
    plot_fn(x32, y32, 'o-', label='float32', color='red')
    plot_fn(x64, y64, 'o-', label='float64', color='blue')
    ax.set_xlabel('Rozmiar macierzy (n)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def plot_results():
    task1_small_float32 = pd.read_csv('data/task1_small_float32.csv')
    task1_small_float64 = pd.read_csv('data/task1_small_float64.csv')
    task1_big_float32   = pd.read_csv('data/task1_big_float32.csv')
    task1_big_float64   = pd.read_csv('data/task1_big_float64.csv')

    specs = [
        (task1_small_float32, task1_small_float64, 'error',            'Błąd maksymalny',        'Błąd rozwiązania — zadanie 1 (małe n)',        'plots/task1_small_error.png',            True),
        (task1_small_float32, task1_small_float64, 'time',             'Czas rozwiązania [s]',   'Czas rozwiązania — zadanie 1 (małe n)',        'plots/task1_small_time.png',             False),
        (task1_small_float32, task1_small_float64, 'condition_number', 'Wskaźnik uwarunkowania', 'Uwarunkowanie macierzy — zadanie 1 (małe n)', 'plots/task1_small_condition.png',        True),
        (task1_big_float32,   task1_big_float64,   'error',            'Błąd maksymalny',        'Błąd rozwiązania — zadanie 1 (duże n)',        'plots/task1_big_error.png',              True),
        (task1_big_float32,   task1_big_float64,   'time',             'Czas rozwiązania [s]',   'Czas rozwiązania — zadanie 1 (duże n)',        'plots/task1_big_time.png',               False),
        (task1_big_float32,   task1_big_float64,   'condition_number', 'Wskaźnik uwarunkowania', 'Uwarunkowanie macierzy — zadanie 1 (duże n)', 'plots/task1_big_condition.png',          True),
    ]

    for df32, df64, col, ylabel, title, path, semilogy in specs:
        make_plot(df32['n'], df32[col], df64['n'], df64[col], ylabel, title, path, semilogy)

    print("Wszystkie wykresy zostały zapisane w katalogu 'plots'")


def run_benchmarks_task1():
    benchmark(task_1_matrix, FIRST_N, PRECISIONS, gauss, "task1_small")
    benchmark(task_1_matrix, SECOND_N, PRECISIONS, gauss, "task1_big")
    plot_results()

In [7]:
run_benchmarks_task1()

Task: task1_small, Precision: float32, n: 2, Error: 2.384186e-07, Cond: 1.800000e+01, Time: 0.000139s
Task: task1_small, Precision: float32, n: 3, Error: 5.960464e-06, Cond: 6.600013e+02, Time: 0.000099s
Task: task1_small, Precision: float32, n: 4, Error: 3.151894e-04, Cond: 2.000029e+04, Time: 0.000141s
Task: task1_small, Precision: float32, n: 5, Error: 1.368284e-03, Cond: 7.985623e+05, Time: 0.000238s
Task: task1_small, Precision: float32, n: 6, Error: 3.294513e-01, Cond: 2.808903e+07, Time: 0.000355s
Task: task1_small, Precision: float32, n: 7, Error: 2.080186e+00, Cond: 1.636898e+09, Time: 0.002102s
Task: task1_small, Precision: float32, n: 8, Error: 3.558672e+01, Cond: 3.866330e+09, Time: 0.000411s
Task: task1_small, Precision: float32, n: 9, Error: 5.161784e+01, Cond: 1.238405e+10, Time: 0.000492s
Task: task1_small, Precision: float32, n: 10, Error: 3.193153e+01, Cond: 4.993668e+09, Time: 0.000414s
Task: task1_small, Precision: float32, n: 11, Error: 3.702252e+02, Cond: 3.357705

Zadanie 2

In [8]:
FIRST_N = list(range(2, 21))
SECOND_N = list(range(2, 200))
PRECISIONS = [np.float32, np.float64]

os.makedirs('data', exist_ok=True)
os.makedirs('plots', exist_ok=True)
pd.set_option('display.float_format', '{:.5f}'.format)


def benchmark_task2(n_range, precisions, method, task_name):
    for precision in precisions:
        precision_name = np.dtype(precision).name
        rows = []

        for n in n_range:
            X = x_vector(n, precision)
            A, b = task_2_matrix(n, X, precision)

            t0 = time.perf_counter()
            X_approx = method(A.copy(), b.copy(), precision)
            solve_time = time.perf_counter() - t0

            error = maximum_error(X, X_approx)
            cond = np.linalg.cond(A, p=1)
            rows.append({'n': n, 'error': error, 'condition_number': cond, 'time': solve_time})

            print(f"Task: {task_name}, Precision: {precision_name}, n: {n}, "
                  f"Error: {error:.6e}, Cond: {cond:.6e}, Time: {solve_time:.6f}s")

        path = f"data/{task_name}_{precision_name}.csv"
        pd.DataFrame(rows).to_csv(path, index=False, float_format='{:.4e}'.format)
        print(f"Results saved to {path}")


def plot_results_task2():
    task2_small_float32 = pd.read_csv('data/task2_small_float32.csv')
    task2_small_float64 = pd.read_csv('data/task2_small_float64.csv')
    task2_big_float32   = pd.read_csv('data/task2_big_float32.csv')
    task2_big_float64   = pd.read_csv('data/task2_big_float64.csv')

    specs = [
        (task2_small_float32, task2_small_float64, 'error',            'Błąd maksymalny',        'Błąd rozwiązania — zadanie 2 (małe n)',        'plots/task2_small_error.png',     True),
        (task2_small_float32, task2_small_float64, 'time',             'Czas rozwiązania [s]',   'Czas rozwiązania — zadanie 2 (małe n)',        'plots/task2_small_time.png',      False),
        (task2_small_float32, task2_small_float64, 'condition_number', 'Wskaźnik uwarunkowania', 'Uwarunkowanie macierzy — zadanie 2 (małe n)', 'plots/task2_small_condition.png', True),
        (task2_big_float32,   task2_big_float64,   'error',            'Błąd maksymalny',        'Błąd rozwiązania — zadanie 2 (duże n)',        'plots/task2_big_error.png',       True),
        (task2_big_float32,   task2_big_float64,   'time',             'Czas rozwiązania [s]',   'Czas rozwiązania — zadanie 2 (duże n)',        'plots/task2_big_time.png',        False),
        (task2_big_float32,   task2_big_float64,   'condition_number', 'Wskaźnik uwarunkowania', 'Uwarunkowanie macierzy — zadanie 2 (duże n)', 'plots/task2_big_condition.png',   True),
    ]

    for df32, df64, col, ylabel, title, path, semilogy in specs:
        make_plot(df32['n'], df32[col], df64['n'], df64[col], ylabel, title, path, semilogy)

    print("Wszystkie wykresy zostały zapisane w katalogu 'plots'")


def run_benchmarks_task2():
    benchmark_task2(FIRST_N, PRECISIONS, gauss, "task2_small")
    benchmark_task2(SECOND_N, PRECISIONS, gauss, "task2_big")
    plot_results_task2()

In [9]:
run_benchmarks_task2()

Task: task2_small, Precision: float32, n: 2, Error: 0.000000e+00, Cond: 3.000000e+00, Time: 0.000291s
Task: task2_small, Precision: float32, n: 3, Error: 5.960464e-08, Cond: 8.666667e+00, Time: 0.000818s
Task: task2_small, Precision: float32, n: 4, Error: 1.192093e-07, Cond: 1.650000e+01, Time: 0.000278s
Task: task2_small, Precision: float32, n: 5, Error: 1.788139e-07, Cond: 2.680000e+01, Time: 0.000399s
Task: task2_small, Precision: float32, n: 6, Error: 7.748604e-07, Cond: 3.966666e+01, Time: 0.000514s
Task: task2_small, Precision: float32, n: 7, Error: 3.337860e-06, Cond: 5.457142e+01, Time: 0.000433s
Task: task2_small, Precision: float32, n: 8, Error: 4.649162e-06, Cond: 7.241667e+01, Time: 0.000344s
Task: task2_small, Precision: float32, n: 9, Error: 4.053116e-06, Cond: 9.238096e+01, Time: 0.000463s
Task: task2_small, Precision: float32, n: 10, Error: 5.245209e-06, Cond: 1.147286e+02, Time: 0.000484s
Task: task2_small, Precision: float32, n: 11, Error: 3.457069e-06, Cond: 1.397829

Zadanie 3

In [10]:
THIRD_N = list(range(2, 200))
PRECISIONS = [np.float32, np.float64]

def benchmark(n_range, precisions, task_name):
    for precision in precisions:
        precision_name = np.dtype(precision).name
        rows = []

        for n in n_range:
            X = x_vector(n, precision)
            A, b = task_3_matrix(n, X, precision)

            t0 = time.perf_counter()
            X_gauss = gauss(A.copy(), b.copy(), precision)
            time_gauss = time.perf_counter() - t0

            t0 = time.perf_counter()
            X_thomas = thomas(A.copy(), b.copy(), precision)
            time_thomas = time.perf_counter() - t0

            error_gauss = maximum_error(X, X_gauss)
            error_thomas = maximum_error(X, X_thomas)
            cond = np.linalg.cond(A, p=1)

            rows.append({
                'n': n,
                'error_gauss': error_gauss,
                'error_thomas': error_thomas,
                'time_gauss': time_gauss,
                'time_thomas': time_thomas,
                'condition_number': cond
            })

            print(f"Task: {task_name}, Precision: {precision_name}, n: {n}, "
                  f"Err_G: {error_gauss:.6e}, Err_T: {error_thomas:.6e}, "
                  f"Time_G: {time_gauss:.6f}s, Time_T: {time_thomas:.6f}s")

        path = f"data/{task_name}_{precision_name}.csv"
        pd.DataFrame(rows).to_csv(path, index=False, float_format='{:.4e}'.format)
        print(f"Results saved to {path}")


def plot_results():
    data = {
        prec: pd.read_csv(f'data/task3_{prec}.csv')
        for prec in ('float32', 'float64')
    }

    plots = [
        ('error_gauss',   'error_thomas',   True,  'Błąd maksymalny',      'Błąd rozwiązania - Gauss vs Thomas'),
        ('time_gauss',    'time_thomas',     False, 'Czas rozwiązania [s]', 'Czas rozwiązania - Gauss vs Thomas'),
        ('condition_number', None,           True,  'Wskaźnik uwarunkowania', 'Uwarunkowanie macierzy z zadania 3'),
    ]

    for prec_name, df in data.items():
        for col_a, col_b, semilogy, ylabel, title in plots:
            fig, ax = plt.subplots(figsize=(10, 6))
            fn = ax.semilogy if semilogy else ax.plot

            fn(df['n'], df[col_a], 'o-', label='Gauss' if col_b else 'float64', color='red')
            if col_b:
                fn(df['n'], df[col_b], 'o-', label='Thomas', color='blue')

            ax.set_xlabel('Rozmiar macierzy (n)')
            ax.set_ylabel(ylabel)
            ax.set_title(f'{title} ({prec_name})')
            ax.legend()
            ax.grid(True)
            ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
            fig.tight_layout()

            col_tag = col_a.replace('_gauss', '').replace('_thomas', '')
            fig.savefig(f'plots/task3_{col_tag}_{prec_name}.png')
            plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 6))
    for prec_name, color in zip(('float32', 'float64'), ('red', 'blue')):
        df = data[prec_name]
        ax.plot(df['n'], df['time_gauss'],   'o-', label=f'Gauss {prec_name}',  color=color)
        ax.plot(df['n'], df['time_thomas'],  's--', label=f'Thomas {prec_name}', color=color, alpha=0.6)
    ax.set_xlabel('Rozmiar macierzy (n)')
    ax.set_ylabel('Czas rozwiązania [s]')
    ax.set_title('Porównanie czasu: Gauss vs Thomas')
    ax.legend()
    ax.grid(True)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    fig.tight_layout()
    fig.savefig('plots/task3_time_comparison.png')
    plt.close(fig)

    print("Wszystkie wykresy zadania 3 zapisane w katalogu 'plots'")


def run_benchmarks_task3():
    benchmark(THIRD_N, PRECISIONS, "task3")
    plot_results()

In [11]:
run_benchmarks_task3()

Task: task3, Precision: float32, n: 2, Err_G: 0.000000e+00, Err_T: 0.000000e+00, Time_G: 0.000161s, Time_T: 0.000031s
Task: task3, Precision: float32, n: 3, Err_G: 0.000000e+00, Err_T: 0.000000e+00, Time_G: 0.000431s, Time_T: 0.000027s
Task: task3, Precision: float32, n: 4, Err_G: 0.000000e+00, Err_T: 0.000000e+00, Time_G: 0.000194s, Time_T: 0.000032s
Task: task3, Precision: float32, n: 5, Err_G: 0.000000e+00, Err_T: 0.000000e+00, Time_G: 0.000152s, Time_T: 0.000036s
Task: task3, Precision: float32, n: 6, Err_G: 0.000000e+00, Err_T: 0.000000e+00, Time_G: 0.000249s, Time_T: 0.000065s
Task: task3, Precision: float32, n: 7, Err_G: 5.960464e-08, Err_T: 5.960464e-08, Time_G: 0.000271s, Time_T: 0.000041s
Task: task3, Precision: float32, n: 8, Err_G: 5.960464e-08, Err_T: 5.960464e-08, Time_G: 0.000602s, Time_T: 0.000056s
Task: task3, Precision: float32, n: 9, Err_G: 1.192093e-07, Err_T: 5.960464e-08, Time_G: 0.000484s, Time_T: 0.000058s
Task: task3, Precision: float32, n: 10, Err_G: 1.192093e